In [19]:
import os
import shutil
from pathlib import Path

import pandas as pd
import torch
import tqdm
from descent.targets.dimers import create_from_des

In [20]:
DES_DIR = Path("/scratch1/joao/joao_vol/DES_datasets/DES370K/")
DES_FILE = DES_DIR / "DES370K.csv"

OUTPUT_DIR = Path("./DES370K/")
FILTERED_FILE = OUTPUT_DIR / "DES370K.csv"

ALLOWED_ELEMENTS = ["H", "C", "F", "Cl", "Br"]

In [21]:
df_filtered

,smiles0,smiles1,charge0,charge1,natoms0,natoms1,system_id,group_orig,group_id,k_index,...,sapt_exdisp_os,sapt_exdisp_ss,sapt_delta_HF,sapt_all,nn_CCSD(T)_all,nn_CCSD(T)_all_05,nn_CCSD(T)_all_95,xyz,elements,elements_lists
4333,CC(C)C,c1ccccc1,0,0,14,12,13981,370K,394922,-8,...,2.80077,4.04403,-6.40749,16.27053,20.91109,19.74752,21.73669,3.51354 3.63949 1.76997 2.72138 2.80546 2.7765...,C C C C H H H H H H H H H H C C C C C C H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
4334,CC(C)C,c1ccccc1,0,0,14,12,13981,370K,394922,-7,...,2.24733,3.25779,-5.56581,9.54661,13.51142,12.47462,14.24802,3.51354 3.63949 1.76997 2.72138 2.80546 2.7765...,C C C C H H H H H H H H H H C C C C C C H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
4335,CC(C)C,c1ccccc1,0,0,14,12,13981,370K,394922,-6,...,1.78992,2.60595,-4.62868,4.71447,7.94877,7.15708,8.61363,3.51354 3.63949 1.76997 2.72138 2.80546 2.7765...,C C C C H H H H H H H H H H C C C C C C H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
4336,CC(C)C,c1ccccc1,0,0,14,12,13981,370K,394922,-5,...,1.41567,2.07198,-3.72997,1.31652,4.12731,3.34600,4.90678,3.51354 3.63949 1.76997 2.72138 2.80546 2.7765...,C C C C H H H H H H H H H H C C C C C C H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
4337,CC(C)C,c1ccccc1,0,0,14,12,13981,370K,394922,-4,...,1.11234,1.63914,-2.93473,-1.00564,1.51818,0.82708,1.97720,3.51354 3.63949 1.76997 2.72138 2.80546 2.7765...,C C C C H H H H H H H H H H C C C C C C H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367519,CCCC,CCCC,0,0,14,14,11136,370K,546178,20,...,0.00004,0.00016,-0.00601,-0.16987,-0.16428,-0.17444,-0.15173,-14.83602 6.10181 16.04491 -13.641 5.41334 15....,C C C C H H H H H H H H H H C C C C H H H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
367520,CCCC,CCCC,0,0,14,14,11136,370K,546178,25,...,0.00001,0.00003,-0.00386,-0.10559,-0.10089,-0.10792,-0.09306,-14.83602 6.10181 16.04491 -13.641 5.41334 15....,C C C C H H H H H H H H H H C C C C H H H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
367521,CCCC,CCCC,0,0,14,14,11136,370K,546178,30,...,0.00000,0.00001,-0.00235,-0.06801,-0.06411,-0.07025,-0.05725,-14.83602 6.10181 16.04491 -13.641 5.41334 15....,C C C C H H H H H H H H H H C C C C H H H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."
367522,CCCC,CCCC,0,0,14,14,11136,370K,546178,35,...,0.00000,0.00000,-0.00154,-0.04539,-0.04319,-0.04768,-0.03854,-14.83602 6.10181 16.04491 -13.641 5.41334 15....,C C C C H H H H H H H H H H C C C C H H H H H ...,"[C, C, C, C, H, H, H, H, H, H, H, H, H, H, C, ..."


In [ ]:
df = pd.read_csv(DES_FILE)

df['elements_lists'] = df['elements'].str.split(" ")
# The anions give problem with NAGL, [H][H] gives problems with interchange.
forbidden = ['[Cl-]', '[F-]', '[Br-]', '[H][H]']
mask = df.apply(
    lambda row: all(elem in ALLOWED_ELEMENTS for elem in row['elements_lists'])
                and row['smiles0'] not in forbidden
                and row['smiles1'] not in forbidden,
    axis=1
)
df_filtered = df[mask]
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Correct the group origin column.
df_filtered['group_orig'] = '370K' 
df_filtered.to_csv(FILTERED_FILE, index=False)

# Remove old geometries if they exist.
geo_output = OUTPUT_DIR / "geometries"
if geo_output.exists():
    shutil.rmtree(geo_output)

# Copy geometries from source
shutil.copytree(DES_DIR / "geometries", geo_output)

/tmp/ipykernel_3225404/4217351117.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['group_orig'] = '370K'


PosixPath('DES370K/geometries')

In [23]:
def energy_fn(group_data, geometry_ids, coords):
    """
    Extract reference energies from the DES metadata CSV (used with DES_DIR).

    Expects a ``reference`` column with energies in kcal/mol.
    """
    return torch.tensor(group_data["cbs_CCSD(T)_all"].values, dtype=torch.float64)

In [24]:
dataset = create_from_des(OUTPUT_DIR, energy_fn=energy_fn)
dataset.save_to_disk(OUTPUT_DIR / "DES370K_dataset/")

loading dimers: 100%|██████████| 194/194 [00:03<00:00, 62.87it/s] 
/home/joaomorado/repos/descent-myfork/descent/targets/dimers.py:68: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "coords": torch.tensor(dimer["coords"]).flatten().tolist(),
/home/joaomorado/repos/descent-myfork/descent/targets/dimers.py:69: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "energy": torch.tensor(dimer["energy"]).flatten().tolist(),


Saving the dataset (0/1 shards):   0%|          | 0/603 [00:00<?, ? examples/s]